# Clean re-run of the sonar detection experiments

I am retraining all four models on the corrected pipeline and scoring them against one ground
truth, one split, and one postprocessing config.

My original comparison could not support its conclusion because three implementation defects each
favoured DCCAN over the baselines I was comparing it against, and the splits leaked 219 of 242
validation images into the labelled training set. I wrote all of that up in `docs/AUDIT.md`. This
notebook is how I re-run the experiment properly.

Before running anything: Runtime, then Change runtime type, then pick A100 or L4.


In [4]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch; print('CUDA available:', torch.cuda.is_available())

NVIDIA A100-SXM4-80GB, 81920 MiB
CUDA available: True


## 1. Mount Drive and clone the repo

Checkpoints go to Drive because Colab wipes `/content` when a session drops, and DCCAN takes
about two hours. The dataset is committed inside the repo, so the clone brings it with it and I
have nothing to upload.


In [5]:
from google.colab import drive; drive.mount('/content/drive')

!git clone -q https://github.com/Kablan-ASBN/sonar-object-detection.git /content/sonar
%cd /content/sonar
!pip install -q -e '.[dev]'

import os
os.makedirs('/content/drive/MyDrive/sonar-runs', exist_ok=True)
os.makedirs('/content/sonar/preds', exist_ok=True)

!du -sh data && git log -1 --oneline

Mounted at /content/drive
/content/sonar
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 116.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 134.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.2/257.2 kB 29.4 MB/s eta 0:00:00
  Building editable for sonar-object-detection (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are in

## 2. Check the splits before I train anything

This has to exit 0. It is the check that would have caught my original leak, and it runs in CI on
every push. If it reports leakage I stop here, because nothing downstream would mean anything.


In [6]:
!sonar audit leakage --root raw=data/line2voc \
                     --root denoised=data/line2voc_preprocessed \
                     --root augmented=data/line2voc_preprocessed_augmented


split sizes:
  raw/train         1429 ids
  raw/val            179 ids
  raw/test           180 ids
  denoised/train    1429 ids
  denoised/val       179 ids
  denoised/test      180 ids
  augmented/train   1429 ids
  augmented/val      179 ids
  augmented/test     180 ids

shared ids (rows: training splits, columns: evaluation splits):
                        raw/val        raw/test    denoised/val   denoised/test   augmented/val  augmented/test
  raw/train                   0               0               0               0               0               0
  denoised/train              0               0               0               0               0               0
  augmented/train             0               0               0               0               0               0

no training id appears in any evaluation split
clean: no training split intersects any evaluation split


## 3. Train the four models

I run these one cell at a time. If the session drops I lose one model, not four. On an A100 the
baselines take roughly 40 minutes each, DANN about 1.5 hours, and DCCAN about 2 hours.

`--mode raw` is not optional. It writes a sidecar recording that no score floor was applied, and
`sonar eval` refuses a filtered file unless I pass `--allow-filtered`. I never pass that flag for a
number I intend to report. Mixing a thresholded export with an unthresholded one is exactly the
defect that made my original baselines look worse than they were.


### `baseline_raw`

Trained on raw sonar, the only model in my original comparison that was not affected by the leak.


In [7]:
CFG = "baseline_raw"
CKPT = f"/content/drive/MyDrive/sonar-runs/{CFG}.pt"
CSV  = f"/content/sonar/preds/{CFG}.csv"

!sonar train --config configs/{CFG}.yaml --out {CKPT} --device cuda

!sonar predict --checkpoint {CKPT} --root data/line2voc --split test \
               --out {CSV} --mode raw --device cuda

!cp {CSV} {CSV}.meta.json /content/drive/MyDrive/sonar-runs/ 2>/dev/null || true

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100% 160M/160M [00:00<00:00, 235MB/s]
/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
epoch 1: loss_box_reg=0.2157 loss_classifier=0.2359 loss_objectness=0.3543 loss_rpn_box_reg=0.4955 total=1.3013 (127.8s) | AP50=0.0686 AP50_object=0.0547 AP50_shadow=0.0825 AP75=0.0014 mAP=0.0154 mAR100=0.1225
epoch 2: loss_box_reg=0.2537 loss_classifier=0.2080 loss_objectness=0.2034 loss_rpn_b

### `baseline_denoised`

Trained on the median filtered source domain.


In [8]:
CFG = "baseline_denoised"
CKPT = f"/content/drive/MyDrive/sonar-runs/{CFG}.pt"
CSV  = f"/content/sonar/preds/{CFG}.csv"

!sonar train --config configs/{CFG}.yaml --out {CKPT} --device cuda

!sonar predict --checkpoint {CKPT} --root data/line2voc --split test \
               --out {CSV} --mode raw --device cuda

!cp {CSV} {CSV}.meta.json /content/drive/MyDrive/sonar-runs/ 2>/dev/null || true

/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
epoch 1: loss_box_reg=0.2242 loss_classifier=0.2394 loss_objectness=0.3536 loss_rpn_box_reg=0.4942 total=1.3114 (126.4s) | AP50=0.0794 AP50_object=0.0563 AP50_shadow=0.1026 AP75=0.0013 mAP=0.0179 mAR100=0.1257
epoch 2: loss_box_reg=0.2579 loss_classifier=0.2083 loss_objectness=0.2021 loss_rpn_box_reg=0.4452 total=1.1135 (35.0s) | AP50=0.1100 AP50_object=0.0798 AP50_shadow=0.1402 AP75=0.0030 mAP=0.0261 mAR100=0.1291
epoch 3: loss_box_reg=0.2655 loss_classifier=0.2018 loss_objectness=0.1887 loss_rpn_box

### `dann`

Global adversarial alignment, now with the domain features going through `model.transform`.


In [9]:
CFG = "dann"
CKPT = f"/content/drive/MyDrive/sonar-runs/{CFG}.pt"
CSV  = f"/content/sonar/preds/{CFG}.csv"

!sonar train --config configs/{CFG}.yaml --out {CKPT} --device cuda

!sonar predict --checkpoint {CKPT} --root data/line2voc --split test \
               --out {CSV} --mode raw --device cuda

!cp {CSV} {CSV}.meta.json /content/drive/MyDrive/sonar-runs/ 2>/dev/null || true

/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
epoch 1: loss_box_reg=0.1746 loss_classifier=0.3284 loss_dann=0.6864 loss_objectness=0.7009 loss_rpn_box_reg=0.5474 total=2.4376 (144.5s) | AP50=0.0185 AP50_object=0.0034 AP50_shadow=0.0336 AP75=0.0007 mAP=0.0039 mAR100=0.0626
epoch 2: loss_box_reg=0.2223 loss_classifier=0.2577 loss_dann=0.6623 loss_objectness=0.2301 loss_rpn_box_reg=0.4725 total=1.8449 (52.2s) | AP50=0.0342 AP50_object=0.0071 AP50_shadow=0.0614 AP75=0.0006 mAP=0.0082 mAR100=0.0813
epoch 3: loss_box_reg=0.2290 loss_classifier=0.2355 l

### `dccan`

The three path architecture i proposed, with the proxy classifier actually trained.


In [10]:
CFG = "dccan"
CKPT = f"/content/drive/MyDrive/sonar-runs/{CFG}.pt"
CSV  = f"/content/sonar/preds/{CFG}.csv"

!sonar train --config configs/{CFG}.yaml --out {CKPT} --device cuda

!sonar predict --checkpoint {CKPT} --root data/line2voc --split test \
               --out {CSV} --mode raw --device cuda

!cp {CSV} {CSV}.meta.json /content/drive/MyDrive/sonar-runs/ 2>/dev/null || true

/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
epoch 1: loss_box_reg=0.1275 loss_cdan=0.6931 loss_classifier=0.2821 loss_dann=0.6895 loss_objectness=0.8707 loss_proposal=0.6541 loss_proxy_aux=0.1593 loss_rpn_box_reg=0.5610 total=4.0373 (132.1s) | AP50=0.0069 AP50_object=0.0025 AP50_shadow=0.0112 AP75=0.0001 mAP=0.0015 mAR100=0.0448
epoch 2: loss_box_reg=0.1747 loss_cdan=0.6931 loss_classifier=0.2516 loss_dann=0.6774 loss_objectness=0.2878 loss_proposal=0.5454 loss_proxy_aux=0.0508 loss_rpn_box_reg=0.5100 total=3.1908 (37.9s) | AP50=0.0124 AP50_obj

## 4. Score all four against the same ground truth

One `--gt-root`, one `--split`, four models, one postprocessing config. This is the comparison my
original one was not: back then CLAHE+Aug was scored against a different dataset root whose
validation split shared only 23 of its ids with the one the other four models used.

`GroundTruth` in this codebase is bound to a single root and a single split, and `compare` takes
exactly one, so mixing them is now a type error rather than something I have to remember.


In [11]:
!sonar eval --gt-root data/line2voc --split test \
  --preds raw=preds/baseline_raw.csv \
  --preds denoised=preds/baseline_denoised.csv \
  --preds dann=preds/dann.csv \
  --preds dccan=preds/dccan.csv \
  --froc /content/drive/MyDrive/sonar-runs/froc.csv

/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
ground truth: root=data/line2voc split=test, 180 images, 2877 boxes
            mAP   AP50   AP75  mAR100  AP50_object  AP50_shadow
model                                                          
raw      0.0403 0.1619 0.0073  0.1592       0.1087       0.2151
dccan    0.0322 0.1299 0.0053  0.1450       0.0939       0.1658
denoised 0.0280 0.1141 0.0034  0.1439       0.0723       0.1559
dann     0.0290 0.1131 0.0060  0.1396       0.0795       0.1468
wrote /content/drive/MyDrive/sonar-runs/froc.csv


## 5. Save everything to Drive

So the predictions and the FROC curve survive the session.


In [12]:
!cp -v preds/*.csv preds/*.meta.json /content/drive/MyDrive/sonar-runs/ 2>/dev/null | tail -5
!ls -la /content/drive/MyDrive/sonar-runs/

'preds/dccan.csv' -> '/content/drive/MyDrive/sonar-runs/dccan.csv'
'preds/baseline_denoised.csv.meta.json' -> '/content/drive/MyDrive/sonar-runs/baseline_denoised.csv.meta.json'
'preds/baseline_raw.csv.meta.json' -> '/content/drive/MyDrive/sonar-runs/baseline_raw.csv.meta.json'
'preds/dann.csv.meta.json' -> '/content/drive/MyDrive/sonar-runs/dann.csv.meta.json'
'preds/dccan.csv.meta.json' -> '/content/drive/MyDrive/sonar-runs/dccan.csv.meta.json'
total 693919
drwx------ 2 root root      4096 Sep 12 03:47 .
drwx------ 8 root root      4096 Sep 12 02:18 ..
-rw------- 1 root root   7223238 Sep 12 03:47 baseline_denoised.csv
-rw------- 1 root root      7586 Sep 12 03:47 baseline_denoised.csv.meta.json
-rw------- 1 root root 165752179 Sep 12 02:58 baseline_denoised.pt
-rw------- 1 root root   7271397 Sep 12 03:47 baseline_raw.csv
-rw------- 1 root root      7586 Sep 12 03:47 baseline_raw.csv.meta.json
-rw------- 1 root root 165750674 Sep 12 02:40 baseline_raw.pt
-rw------- 1 root root   727

## How I plan to read the result

Four corrections all move against DCCAN compared to my original run:

1. the baselines are no longer score floored, so they keep the low confidence tail that COCO AP
   integrates over
2. the baselines and DANN no longer train on mirrored images with unmirrored boxes
3. DANN's discriminator now sees the same normalised, resized input the detector sees
4. nothing is evaluated on images it was trained on

My original margin was 0.011 AP50, and each of those corrections is larger than that. So I expect
the gap to shrink and I would not be surprised if it disappears. If the baselines come out ahead,
that is the result and I will report it. I am not going to tune until DCCAN wins, because that is
how the first result happened.

The one claim from the original work I still stand behind either way: the three path design trains
stably under mixed precision, where a standalone CDAN outer product mapping did not.
